# CAMUS — DRL Trial Run: hard-mining Tier 1/2 + debris/SectorPool fixes (LV_endo, Phase B)

**Purpose of this notebook**: a short, cheap trial run to validate three fixes made this
session, all stacked together, on the exact class/agent combo we diagnosed in depth
(Phase B CAMUS LV_endo, DuelingDDQN):

1. **Debris-channel fix** — state channels 4-5 now show only the largest connected
   component of the U-Net init, matching what the single closed contour can represent
   (previously debris blobs leaked into the state and biased the policy).
2. **SectorPool fix** — the spatial head's angular wedges are now centred on each
   sample's own mask centroid instead of a fixed image centre (previously 54% of
   feature-map cells disagreed with the env's own sector convention for off-centre masks).
3. **Hard-mining Tier 1 (boundary-aware, always on) + Tier 2 (adaptive, opt-in here)** —
   the exponential hard-mining weight is now gated by `boundary_frac` (so samples that
   are "hard" because of unfixable topology error stop getting amplified just for
   looking bad) and, with `hard_mining_adaptive=True` below, recomputed periodically
   from each sample's own demonstrated progress instead of staying frozen at its
   step-0 value for the whole run.

**Baseline to compare against** (same class/agent, BEFORE these fixes, from the log
analysed earlier this session): test set `init_dice_mean=0.9129 -> final_dice_mean=0.9067`
(delta -0.0061), `value_floored_dice_mean=0.9113` (delta -0.0016), STOP-action rate 12%,
action-usage histogram dominated by `out7`/`in2`/`out2` almost regardless of sample.

This run uses a **short trial budget** (`TRAIN_STEPS=8000`, not the full 30k) — the goal
is to check the fixes behave mechanically as designed (section 5 diagnostics) and see an
early trend, not to produce a final publishable number. Re-run with the full budget once
the trial looks healthy.

| Variable | Options |
|---|---|
| `CAMUS_CLASS` | fixed to `'c1'` (LV_endo) for direct comparison to the diagnosed baseline |
| `AGENT_NAME` | fixed to `'DuelingDDQN'` — cheapest agent, richest diagnostic history this session |

> ### PHASE B NOTEBOOK -- LOW-DATA REGIME (CAMUS)
> Refines a lite U-Net retrained on a ~10% patient subset (~148/1500 training images) --
> see `notebooks/phaseB/unet/01_camus_lite.ipynb`. Attach the Phase-B checkpoint dataset,
> not the Phase-A one, or the auto-detect below raises `FileNotFoundError`.


## 0 - Install + locate package

In [ ]:
import subprocess, sys
from pathlib import Path

init_files = list(Path('/kaggle/input').rglob('iteris/__init__.py'))
if not init_files:
    raise RuntimeError('iteris-pkg not attached.')
PKG_ROOT = init_files[0].parent.parent
subprocess.run(['pip', 'install', '-r', str(PKG_ROOT / 'requirements.txt'),
                '--quiet', '--upgrade'], check=True)
sys.path.insert(0, str(PKG_ROOT))
print(f'PKG_ROOT = {PKG_ROOT}')


## 1 - Configure - set algorithm here

In [ ]:
# =============================================================================
CAMUS_CLASS = 'c1'           # fixed for this trial -- direct comparison to the diagnosed baseline
AGENT_NAME  = 'DuelingDDQN'  # fixed for this trial -- cheapest agent, richest diagnostic history
# =============================================================================

_CONFIG_MAP = {'c1': 'camus_drl_c1.yaml',
               'c2': 'camus_drl_c2.yaml',
               'c3': 'camus_drl_c3.yaml'}
assert CAMUS_CLASS in _CONFIG_MAP, f'CAMUS_CLASS must be one of {list(_CONFIG_MAP)}'

from iteris.config import load_drl_class_config, resolve_agent_config, load_config, apply_refinement_config
from iteris.utils  import get_device

cfg_full     = load_drl_class_config(str(PKG_ROOT / 'configs' / 'CAMUS' / 'DRL' / _CONFIG_MAP[CAMUS_CLASS]))
cfg          = resolve_agent_config(cfg_full, AGENT_NAME)

apply_refinement_config(cfg, baseline_cfg_name='CAMUS/camus.yaml', uncertainty_gate=False)

# -- TRIAL: opt into Tier 2 adaptive hard-mining (Tier 1 boundary-awareness is --------
# always on whenever hard_mining_scale>0, no flag needed -- see iteris/drl_training.py).
# Tier 2 recomputes sampling weights from each sample's demonstrated progress every
# `hard_mining_refresh_every` steps instead of freezing them at their step-0 value.
cfg['hard_mining_adaptive']      = True
cfg['hard_mining_refresh_every'] = 1000   # finer than this class's default eval_every (5000) --
                                          # want several refreshes visible within the short trial budget

# -- STOP-bonus sweep (optimal-stopping incentive, see ContourRefineEnv._terminal_step) --
# config.py sets terminal_bonus_scale=20.0 by default; sweep it here to validate the
# magnitude before trusting it in the full runs. Watch STOP-action rate (section 9
# behaviour plot) climb from the ~12% baseline, and deploy(vfloor) close toward best-seen,
# WITHOUT the agent collapsing to stop-immediately (that would show as ~0 mean steps and
# no refinement). Try 10 / 20 / 40:
cfg['terminal_bonus_scale'] = 20.0

baseline_cfg = load_config(str(PKG_ROOT / 'configs' / cfg['baseline_cfg_name']))

# =============================================================================
# PHASE B override -- must match the retrained low-data lite/attention
# checkpoints (~10% patient subset, ~148/1500 training images). This single
# line does TWO things: (1) makes the checkpoint auto-detect below look for
# the _lf10-suffixed checkpoint instead of the full-data one, and (2) makes
# precompute_init_masks() below use the SAME low-data patient subset the
# U-Net was trained on -- required for a fair, leak-free comparison (see
# docs/EXPERIMENTS.md). Do not remove; do not set independently of the
# baseline notebook's label_frac.
baseline_cfg['label_frac'] = 0.10
# =============================================================================

cfg['checkpoint_dir'] = '/kaggle/working'

# Auto-detect CAMUS data root
camus_roots = [p for p in Path('/kaggle/input').rglob('CAMUS') if p.is_dir()]
if camus_roots:
    baseline_cfg['data_root'] = str(camus_roots[0])

# Auto-detect the lite-vs-attention checkpoint by deriving the expected
# filename from baseline_cfg (model_suffix handles _lite_unet vs '' default).
from iteris.utils import model_suffix
_ckpt_name  = f"{baseline_cfg['dataset'].lower()}{model_suffix(baseline_cfg)}_best.pt"
_ckpt_cands = list(Path('/kaggle/input').rglob(_ckpt_name))
if _ckpt_cands:
    cfg['baseline_checkpoint'] = str(_ckpt_cands[0])
else:
    raise FileNotFoundError(
        f'Baseline checkpoint {_ckpt_name!r} not found under /kaggle/input. '
        f'Attach the dataset containing it, or override cfg["baseline_checkpoint"] manually.')

print(f'Agent           : {cfg["agent_type"]}')
print(f'Target class    : {cfg["target_class"]} ({cfg["class_name"]})')
print(f'Reward mode     : {cfg["reward_mode"]}  '
      f'(alpha={cfg.get("reward_alpha","-")} beta={cfg.get("reward_beta","-")} '
      f'hd_norm={cfg.get("hd_norm","-")})')
print(f'Hard mining     : scale={cfg.get("hard_mining_scale")}  '
      f'adaptive={cfg["hard_mining_adaptive"]}  refresh_every={cfg["hard_mining_refresh_every"]}')
print(f'STOP bonus      : terminal_bonus_scale={cfg.get("terminal_bonus_scale")}')
print(f'CAMUS data root : {baseline_cfg["data_root"]}')
print(f'Baseline ckpt   : {cfg["baseline_checkpoint"]}')
print(f'Train steps     : {cfg["train_steps"]}')
print(f'Device          : {get_device()}')


## 2 - Warm-start - pre-compute U-Net init masks
~5 min for CAMUS. Caches U-Net predictions so the DRL loop never re-runs the backbone.

In [ ]:
from iteris.warm_start import precompute_init_masks
import numpy as np

train_samples, val_samples, test_samples = precompute_init_masks(
    baseline_cfg        = baseline_cfg,
    baseline_checkpoint = cfg['baseline_checkpoint'],
    target_class        = cfg['target_class'],
    min_area_fraction   = cfg.get('min_area_fraction', 0.01),
)
print(f'\nSamples - train: {len(train_samples)}  val: {len(val_samples)}  test: {len(test_samples)}')

init_dices = [float((s['init_mask'] & s['gt_mask']).sum() * 2 /
                    (s['init_mask'].sum() + s['gt_mask'].sum() + 1e-6))
              for s in val_samples]
print(f'U-Net init Dice on val ({cfg["class_name"]}): '
      f'mean {np.mean(init_dices):.4f}  median {np.median(init_dices):.4f}')


## 3 - OPTIONAL - Dry-run sanity check (~3 min)
Self-contained `deepcopy(cfg)` - does NOT mutate `cfg`, `train_samples`, or `val_samples`.

In [ ]:
# =============================================================================
RUN_DRY_RUN = False   # Set True for a quick ~2-3 min sanity check before full training
# =============================================================================

if RUN_DRY_RUN:
    import copy
    from iteris.drl_training import run_drl_training

    # Short sanity run on a small train/val slice - same loop as the full run.
    _dry = copy.deepcopy(cfg)
    _dry.update(dict(train_steps=600, prefill_steps=500, buffer_size=3000,
                     eval_every=200, epsilon_decay_steps=400, batch_size=32,
                     hard_mining_refresh_every=200))
    _dry_result = run_drl_training(_dry, train_samples[:30], val_samples[:10])
    print(f'\nDry run passed. Best val score (meaningless at 600 steps): {_dry_result["best_dice"]:.4f}')
    print(f'  cfg["train_steps"] still = {cfg["train_steps"]}  (unchanged)')
    print('  -> Safe to run section 4 below for the real training run.')
else:
    print('Dry run skipped (RUN_DRY_RUN = False). Proceed directly to section 4.')


## 4 - Trial training (short budget)

**DuelingDDQN** discrete contour refinement. This is a TRIAL budget (~8k steps, ~25-40 min
on a T4) - enough to see the hard-mining diagnostics (section 5) develop and an early curve
trend (section 10), not a final result. Re-run with `TRAIN_STEPS=30000` (matching the
diagnosed baseline run) once this trial looks healthy.

The best-val checkpoint is saved to `/kaggle/working` whenever the score improves.

In [ ]:
# -- Diagnostic / quota controls --------------
TRAIN_STEPS = 8000      # TRIAL budget -- validates the fixes mechanically; NOT the full 30k comparison run
RESUME      = False
if TRAIN_STEPS is not None:
    cfg['train_steps'] = TRAIN_STEPS
cfg['resume'] = RESUME
cfg['eval_every'] = 2000   # finer than this class's default (5000) so an 8k-step trial still shows a curve
print(f"[run] train_steps={cfg['train_steps']}  resume={cfg['resume']}  eval_every={cfg['eval_every']}")
# ----------------

from iteris.drl_training import run_drl_training

result    = run_drl_training(cfg, train_samples, val_samples)
agent     = result['agent']
history   = result['history']     # pandas DataFrame
best_dice = result['best_dice']

print(f"\nTraining complete - {cfg['agent_type']} on {cfg['class_name']}")
print(f"  Best val final-Dice : {best_dice:.4f}")
print(f"  Checkpoint          : {result['checkpoint']}")


## 5 - Hard-mining diagnostics - did Tier 1/2 actually work?

Left panel (Tier 1): sampling weight vs. init Dice, coloured by `boundary_frac`. A
correctly-working fix shows LOW weight for red points (topology-broken, boundary_frac
near 0) even at low init Dice - the old formula would have given those the HIGHEST
weight of all, purely for looking bad.

Right panel (Tier 2, since `hard_mining_adaptive=True` above): weight vs. demonstrated
progress. A sample the agent has made real progress on should show LOWER weight than an
equally-hard sample where zero progress has been made - the schedule should stop
"insisting" on a sample once it's solved, and keep insisting on ones that are genuinely
stuck.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

diag = result.get('hard_mining_diag')
if diag is None:
    print('hard_mining_scale<=0 for this config - no diagnostics to show.')
else:
    init_dice = diag['init_dice']
    boundary  = diag['boundary_frac']
    best_seen = diag['best_seen_dice']
    weights   = diag['final_weights']
    progress  = np.maximum(best_seen - init_dice, 0.0)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    sc0 = axes[0].scatter(init_dice, weights, c=boundary, cmap='RdYlGn', s=14, vmin=0, vmax=1)
    axes[0].set(xlabel='init Dice', ylabel='final sampling weight (normalised)',
                title='Tier 1: weight vs init Dice, coloured by boundary_frac\n'
                      '(green=boundary-fixable - OK to boost; red=topology-broken - should NOT be)')
    plt.colorbar(sc0, ax=axes[0], label='boundary_frac')
    axes[0].grid(alpha=0.3)

    if diag['adaptive']:
        sc1 = axes[1].scatter(progress, weights, c=init_dice, cmap='viridis', s=14)
        axes[1].set(xlabel='progress = max(best_seen_dice - init_dice, 0)',
                    ylabel='final sampling weight',
                    title='Tier 2: weight vs demonstrated progress\n'
                          '(expect high-progress "solved" samples to have LOWER weight)')
        plt.colorbar(sc1, ax=axes[1], label='init Dice')
    else:
        axes[1].text(0.5, 0.5, 'hard_mining_adaptive=False\n(Tier 2 off - weights are static)',
                    ha='center', va='center', transform=axes[1].transAxes)
        axes[1].set_title('Tier 2 (not enabled)')
    axes[1].grid(alpha=0.3)

    plt.suptitle(f"{cfg['dataset']} {cfg['agent_type']} {cfg['class_name']} - hard-mining diagnostics")
    plt.tight_layout()
    out = f"/kaggle/working/{cfg['dataset'].lower()}_{cfg['agent_type'].lower()}_hardmining_diag.png"
    plt.savefig(out, dpi=150); plt.show()
    print(f'Saved to {out}')

    n_topo_suppressed = int(((boundary < 0.3) & (init_dice < 0.85)).sum())
    n_progress_made    = int((progress > 0.02).sum())
    print(f'\nSamples that look hard (init_dice<0.85) but are mostly topology-broken '
          f'(boundary_frac<0.3): {n_topo_suppressed}/{len(init_dice)}')
    print(f'Samples with real demonstrated progress (>0.02 Dice gained at some point): '
          f'{n_progress_made}/{len(init_dice)}')


## 6 - Visualisation setup
Defines `replay_episode()` and `ENV_KW`. Used by all cells below.

In [ ]:
import matplotlib.pyplot as plt   # needed by the plot cells below
from iteris.refinement_viz import (
    refinement_env_kwargs, refinement_env_cls, build_replays, plot_comparison,
    plot_playback, plot_behaviour, evaluate_testset)

ENV_KW  = refinement_env_kwargs(cfg)
ENV_CLS = refinement_env_cls(cfg)   # SegmentationEnv, or ContourRefineEnv for contour/TD3 -- auto-detection
                                     # by agent.num_actions silently picks the WRONG env for continuous
                                     # agents (TD3 has no num_actions), so pass it explicitly.
N_VIZ   = 8
replays = build_replays(agent, val_samples, ENV_KW, n_viz=N_VIZ, seed=0, env_cls=ENV_CLS)

CLASS_NAME = cfg.get('class_name', '')
print(f'Built {len(replays)} greedy replays for visualisation.')


## 7 - Sample comparison - best / median / worst

In [ ]:
import matplotlib.pyplot as plt
fig = plot_comparison(
    replays, baseline_cfg, cfg,
    class_idx=cfg.get('target_class', 1), class_name=CLASS_NAME,
    out_path=f"/kaggle/working/{cfg['dataset'].lower()}_{cfg['agent_type'].lower()}_comparison.png")
plt.show()


## 8 - Iteration playback - all steps on best-gain sample
Green contour = GT boundary. This is the UI Iteration Playback page data.

In [ ]:
import matplotlib.pyplot as plt
# Playback the BEST-gain episode step-by-step
fig = plot_playback(
    replays[-1], cfg, class_name=CLASS_NAME,
    out_path=f"/kaggle/working/{cfg['dataset'].lower()}_{cfg['agent_type'].lower()}_playback.png")
plt.show()


## 9 - Behaviour analysis - trajectories + action distribution

In [ ]:
import matplotlib.pyplot as plt
fig = plot_behaviour(
    replays, cfg, class_name=CLASS_NAME,
    out_path=f"/kaggle/working/{cfg['dataset'].lower()}_{cfg['agent_type'].lower()}_behaviour.png")
plt.show()


## 10 - Learning curves

In [ ]:
import matplotlib.pyplot as plt

# history is a DataFrame with columns: step, init_dice_mean, final_dice_mean,
# best_dice_mean, final_hd95_mean, delta_dice_mean
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['step'], history['init_dice_mean'],  label='Init Dice (U-Net)', ls='--', alpha=0.6, color='gray')
ax1.plot(history['step'], history['final_dice_mean'], label=f'Final Dice ({cfg["agent_type"]})', color='C0')
if 'best_dice_mean' in history.columns:
    ax1.plot(history['step'], history['best_dice_mean'], label='Best-seen Dice (ceiling)', color='C2', alpha=0.7)
ax1.set(xlabel='Training step', ylabel='Mean Val Dice',
        title=f"{cfg['dataset']} - {cfg['class_name']} refinement curve (TRIAL)")
ax1.legend(); ax1.grid(alpha=0.3)

delta = history['final_dice_mean'] - history['init_dice_mean']
ax2.plot(history['step'], delta, color='C0')
ax2.axhline(0, color='k', lw=0.8)
ax2.fill_between(history['step'], delta, 0, where=(delta > 0), alpha=0.15, color='green')
ax2.fill_between(history['step'], delta, 0, where=(delta < 0), alpha=0.15, color='red')
ax2.set(xlabel='Training step', ylabel='Delta Dice (final - init)',
        title=f'Refinement gain - {cfg["agent_type"]} ({cfg["class_name"]})')
ax2.grid(alpha=0.3)

plt.suptitle(f"{cfg['dataset']} {cfg['agent_type']} - {cfg['class_name']} learning curves (TRIAL)")
plt.tight_layout()
out = f"/kaggle/working/{cfg['dataset'].lower()}_{cfg['agent_type'].lower()}_curves.png"
plt.savefig(out, dpi=150); plt.show(); print(f'Saved to {out}')


## 11 - Test-set evaluation + summary JSON
Full test set (~7 min).

**Compare against the pre-fix baseline**: `init_dice_mean=0.9129 -> final_dice_mean=0.9067`
(delta -0.0061), `value_floored_dice_mean=0.9113` (delta -0.0016).

In [ ]:
import json
test_metrics = evaluate_testset(agent, test_samples, ENV_KW, env_cls=ENV_CLS,
    refinable_gate=cfg.get('refinable_gate', False),
    refinable_min_cc_frac=cfg.get('refinable_min_cc_frac', 0.004),
    refinable_min_dominance=cfg.get('refinable_min_dominance', 0.5))
print(json.dumps(test_metrics, indent=2))

summary = {**test_metrics,
           'agent_type':    cfg['agent_type'],
           'target_class':  cfg['target_class'],
           'class_name':    cfg['class_name'],
           'dataset':       cfg['dataset'],
           'paradigm':      'local_mask_refinement',
           'trial':         True,
           'hard_mining_adaptive': cfg.get('hard_mining_adaptive', False)}
out = f"/kaggle/working/{cfg['dataset'].lower()}_{cfg['agent_type'].lower()}_test_results.json"
with open(out, 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved to', out)
print(f"\nBaseline (U-Net) Dice: {test_metrics['init_dice_mean']:.4f}")
print(f"Refined  (agent)  Dice: {test_metrics['final_dice_mean']:.4f}  "
      f"(delta {test_metrics['delta_dice_mean']:+.4f})")
print(f"Best-seen ceiling Dice: {test_metrics['best_dice_mean']:.4f}")
print(f"\n--- vs pre-fix baseline ---")
print(f"Baseline run : init=0.9129  final=0.9067 (d-0.0061)  deploy=0.9113 (d-0.0016)")
print(f"This trial   : init={test_metrics['init_dice_mean']:.4f}  "
      f"final={test_metrics['final_dice_mean']:.4f} (d{test_metrics['delta_dice_mean']:+.4f})  "
      f"deploy={test_metrics['value_floored_dice_mean']:.4f} "
      f"(d{test_metrics['value_floored_delta_mean']:+.4f})")
